# Akan (Asante Twi) Text-to-Speech — Live Demo
**Author:** Abdul Rashid Dickson · UG MSc selection, stage 2

Fine-tuned **facebook/mms-tts-aka** (VITS, Meta MMS) on a single-narrator subset of GhanaNLP's *ghana-speech* Asante Twi corpus (CC-BY-NC-4.0).
Runs in **Google Colab** or **desktop Jupyter**. If the fine-tuned checkpoint is not in scope, it falls back to the baseline so the flow always works.

▶ Runtime → Change runtime type → **T4 GPU** (Colab) — CPU also works, just slower.

In [1]:
# 1. Setup (first run installs quietly - WAIT for "transformers ready": ~1-2 min in Colab, ~10 s locally)
%pip install -q transformers soundfile numpy
import numpy as np, soundfile as sf, torch, time
from transformers import VitsModel, VitsTokenizer
print('transformers ready - now run the next cell')


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: E:\UG-TTS-Task\.venv\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


E:\UG-TTS-Task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers ready - now run the next cell


E:\UG-TTS-Task\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
# 2. Load the model (FIRST Colab run downloads ~145 MB: 30-60 s, then cached in the session)
import os, sys, torch, glob, pathlib
os.environ['HF_HUB_DISABLE_XET'] = '1'

FINETUNED_HUB = 'Dickson32-cell/akan-twi-mms'   # auto-detected after training completes
CANDIDATES = []
if os.path.isdir('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_ms=30000)
        hits = glob.glob('/content/drive/MyDrive/UG_TTS/akan-finetuned*')
        if hits: CANDIDATES.append(hits[0])
    except Exception:
        pass
CANDIDATES += [FINETUNED_HUB,
               'E:/UG-TTS-Task/05_models/akan-finetuned',
               'E:/UG-TTS-Task/05_models/mms-tts-aka',
               'facebook/mms-tts-aka']

MODEL_PATH, LABEL = None, None
for cand in CANDIDATES:
    try:
        ok = cand == FINETUNED_HUB or cand.startswith('facebook/') or os.path.isdir(cand)
        if not ok: continue
        tok = VitsTokenizer.from_pretrained(cand)
        model = VitsModel.from_pretrained(cand)
        kind = 'FINE-TUNED' if 'finetuned' in cand.lower() else 'BASELINE'
        MODEL_PATH, LABEL = cand, kind + ' [' + str(cand) + ']'
        break
    except Exception as e:
        print('skip', cand, '->', str(e)[:70])
assert MODEL_PATH is not None, 'no model available - check internet and re-run'
model.eval(); SR = model.config.sampling_rate
torch.manual_seed(42)  # same text -> same audio every run (deterministic demo)
print('LOADED:', LABEL)

skip Dickson32-cell/akan-twi-mms -> Dickson32-cell/akan-twi-mms is not a local folder and is not a valid m


Loading weights: 100%|██████████| 762/762 [00:00<00:00, 3851.13it/s]


LOADED: BASELINE [E:/UG-TTS-Task/05_models/mms-tts-aka]


## Text normalization (own work)
The MMS-aka tokenizer knows only 30 lowercase Akan symbols, so our normalizer (same code as training time) folds case/punctuation, expands digits to spoken Twi numerals, and strips tone marks. Show it working:

In [3]:
# 3. Text normalization (own work) - load the SAME module the training pipeline uses.
# Priority: local repo copy; else download the exact file from GitHub (single source of truth).
import importlib, os, sys, pathlib

def _load_normalize():
    # local candidates
    here = pathlib.Path.cwd()
    for cand in [here/"src"/"normalize_twi.py", here/"normalize_twi.py",
                 pathlib.Path("E:/UG-TTS-Task/repo/src/normalize_twi.py")]:
        if cand.exists():
            sys.path.insert(0, str(cand.parent))
            import normalize_twi; importlib.reload(normalize_twi)
            return normalize_twi.normalize_twi, str(cand)
    # Colab: fetch from GitHub (pinned repo content)
    import urllib.request
    url = "https://raw.githubusercontent.com/Dickson32-cell/akan-tts-asante/master/src/normalize_twi.py"
    pathlib.Path("normalize_twi.py").write_text(urllib.request.urlopen(url, timeout=60).read().decode("utf-8"), encoding="utf-8")
    sys.path.insert(0, str(pathlib.Path("normalize_twi.py").resolve().parent))
    import normalize_twi; importlib.reload(normalize_twi)
    return normalize_twi.normalize_twi, url

normalize_twi, SRC = _load_normalize()
print("normalizer loaded from:", SRC)
print()
for t in ["1 BERƐSOSƐM 1.", "Mema wo akye, me nuanom!", "Afe 2026 mu...", "Ntonto a ɛtɔ so 4 no bɔɔ Seorim."]:
    print(f"{t!r:46} -> {normalize_twi(t)!r}")
print()
print("The tokenizer only knows these 30 symbols: a b d e f g h i k l m n o p r s t u w y ɛ ɔ ' - (+2 unused) —")
print("every normalized string above is guaranteed inside that inventory (unit-tested in repo).")

normalizer loaded from: E:\UG-TTS-Task\repo\src\normalize_twi.py

'1 BERƐSOSƐM 1.'                               -> 'baako berɛsosɛm baako'
'Mema wo akye, me nuanom!'                     -> 'mema wo akye me nuanom'
'Afe 2026 mu...'                               -> 'afe mu'
'Ntonto a ɛtɔ so 4 no bɔɔ Seorim.'             -> 'ntonto a ɛtɔ so ɛnan no bɔɔ seorim'

The tokenizer only knows these 30 symbols: a b d e f g h i k l m n o p r s t u w y ɛ ɔ ' - (+2 unused) —
every normalized string above is guaranteed inside that inventory (unit-tested in repo).


In [4]:
# 4. Synthesize + play inline. Prompts load from the REPO (single source of truth):
# local copy first, GitHub fallback (so the Colab copy always matches the repo).
from pathlib import Path
import urllib.request
_cand = [Path('scripts/demo_prompts.txt'), Path('../scripts/demo_prompts.txt'), Path('demo_prompts.txt')]
_src = next((c for c in _cand if c.exists()), None)
if _src is None:
    Path('scripts').mkdir(exist_ok=True)
    _u = 'https://raw.githubusercontent.com/Dickson32-cell/akan-tts-asante/master/scripts/demo_prompts.txt'
    Path('scripts/demo_prompts.txt').write_text(urllib.request.urlopen(_u, timeout=60).read().decode('utf-8'), encoding='utf-8')
    _src = Path('scripts/demo_prompts.txt')
VETTED = [l.strip() for l in _src.read_text(encoding='utf-8').splitlines() if l.strip() and not l.startswith('#')]
print(f'loaded {len(VETTED)} vetted prompts from {_src}')

def speak(text, save_only=False, tag='demo'):
    norm = normalize_twi(text)
    inputs = tok(text=norm, return_tensors='pt')
    with torch.no_grad():
        wav = model(**inputs).waveform[0]
    fname = f'{tag}_{time.strftime("%H%M%S")}.wav'
    sf.write(fname, (wav.numpy()*32767).astype(np.int16), SR, subtype='PCM_16')
    print(f'[{LABEL[:40]}] "{text}"  ({len(wav)/SR:.1f}s) -> {fname}')
    if not save_only:
        import IPython.display as D
        return D.Audio(fname)

for i, p in enumerate(VETTED, 1): print(f'  {i}. {p}')
try:
    raw = input('Twi [number or text] > ').strip()
except EOFError:
    raw = ''
if raw.isdigit() and 1 <= int(raw) <= len(VETTED): text = VETTED[int(raw)-1]
elif raw: text = raw
else: text = VETTED[0]
speak(text)

  1. Mema wo akye, me nuanom. Wo ho te sɛn?
  2. Sukuu no mu suani no bɔɔ mfomsoɔ, enti ne kyereɛ gyinaa hɔ.
  3. Ɔkraman no suro wia na ɛnnyɛ nea ɔhunu no.
  4. Afe 2026 mu no, sika a yɛde bɛboa mpɔtam hɔfoɔ bɛyɛ aduonu num.
  5. Adwuma pa, akoma pa — na ɛno na ɛkyerɛ obi mu nimdeɛ.


Twi [number or text] >  1


[BASELINE [E:/UG-TTS-Task/05_models/mms-t] "Mema wo akye, me nuanom. Wo ho te sɛn?"  (2.7s) -> demo_213938.wav


In [6]:
# 5. Batch: the five+ deliverable examples (unseen text, with texts printed)
for i, p in enumerate(VETTED, 1):
    print(f'--- Example {i}: {p}')
    a = speak(p, tag=f'deliverable_{i:02d}')
    if a is not None:
        IPython.display.display(a) if 'IPython' in globals() else None
import IPython
IPython.display.display()

--- Example 1: Mema wo akye, me nuanom. Wo ho te sɛn?
[BASELINE [E:/UG-TTS-Task/05_models/mms-t] "Mema wo akye, me nuanom. Wo ho te sɛn?"  (2.6s) -> deliverable_01_213953.wav


--- Example 2: Sukuu no mu suani no bɔɔ mfomsoɔ, enti ne kyereɛ gyinaa hɔ.
[BASELINE [E:/UG-TTS-Task/05_models/mms-t] "Sukuu no mu suani no bɔɔ mfomsoɔ, enti ne kyereɛ gyinaa hɔ."  (4.5s) -> deliverable_02_213955.wav


--- Example 3: Ɔkraman no suro wia na ɛnnyɛ nea ɔhunu no.
[BASELINE [E:/UG-TTS-Task/05_models/mms-t] "Ɔkraman no suro wia na ɛnnyɛ nea ɔhunu no."  (3.4s) -> deliverable_03_213957.wav


--- Example 4: Afe 2026 mu no, sika a yɛde bɛboa mpɔtam hɔfoɔ bɛyɛ aduonu num.
[BASELINE [E:/UG-TTS-Task/05_models/mms-t] "Afe 2026 mu no, sika a yɛde bɛboa mpɔtam hɔfoɔ bɛyɛ aduonu num."  (4.1s) -> deliverable_04_213959.wav


--- Example 5: Adwuma pa, akoma pa — na ɛno na ɛkyerɛ obi mu nimdeɛ.
[BASELINE [E:/UG-TTS-Task/05_models/mms-t] "Adwuma pa, akoma pa — na ɛno na ɛkyerɛ obi mu nimdeɛ."  (4.9s) -> deliverable_05_214001.wav


### Notes for the defense
- The five deliverable examples are **unseen text** — none appear in the training corpus (they are everyday conversational Twi; the corpus is Bible prose).
- Model license: CC-BY-NC-4.0 (inherited) — training harness MIT, dataset CC-BY-NC-4.0; attribution in the repo README + ATTRIBUTION.md.
- Repo: https://github.com/Dickson32-cell/akan-tts-asante